In [1]:
import pandas as pd
import numpy as np
print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [3]:
# Cargar dataset
df = pd.read_csv('../data/processed/tmdb_clean.csv')
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

Dataset cargado: 3229 filas, 18 columnas


,budget,genres,id,keywords,original_language,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,title,vote_average,vote_count,release_year,net_profit,roi
0,237000000,"['Action', 'Adventure', 'Fantasy', 'Science Fi...",19995,"['culture clash', 'future', 'space war', 'spac...",en,Avatar,150.437577,"['Ingenious Film Partners', 'Twentieth Century...","['United States of America', 'United Kingdom']",2009-12-10,2787965087,162,Avatar,7.2,11800,2009,2550965087,1076.36
1,300000000,"['Adventure', 'Fantasy', 'Action']",285,"['ocean', 'drug abuse', 'exotic island', 'east...",en,Pirates of the Caribbean: At World's End,139.082615,"['Walt Disney Pictures', 'Jerry Bruckheimer Fi...",['United States of America'],2007-05-19,961000000,169,Pirates of the Caribbean: At World's End,6.9,4500,2007,661000000,220.33
2,245000000,"['Action', 'Adventure', 'Crime']",206647,"['spy', 'based on novel', 'secret agent', 'seq...",en,Spectre,107.376788,"['Columbia Pictures', 'Danjaq', 'B24']","['United Kingdom', 'United States of America']",2015-10-26,880674609,148,Spectre,6.3,4466,2015,635674609,259.46
3,250000000,"['Action', 'Crime', 'Drama', 'Thriller']",49026,"['dc comics', 'crime fighter', 'terrorist', 's...",en,The Dark Knight Rises,112.312950,"['Legendary Pictures', 'Warner Bros.', 'DC Ent...",['United States of America'],2012-07-16,1084939099,165,The Dark Knight Rises,7.6,9106,2012,834939099,333.98
4,260000000,"['Action', 'Adventure', 'Science Fiction']",49529,"['based on novel', 'mars', 'medallion', 'space...",en,John Carter,43.926995,['Walt Disney Pictures'],['United States of America'],2012-03-07,284139100,132,John Carter,6.1,2124,2012,24139100,9.28


In [6]:
# Se convierten las columnas con listas strings a listas nativas
import ast 

list_columns = ['genres', 'keywords', 'production_companies', 'production_countries']
for col in list_columns: 
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

print("Columnas convertidas correctamente")
print(f"Ejemplo: Genres - {df['genres'].iloc[0]}")
print(f"Tipo de dato: {type(df['genres'].iloc[0])}")

Columnas convertidas correctamente
Ejemplo: Genres - ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
Tipo de dato: <class 'list'>


In [7]:
# Analisis de los generos mas frecuentes
from collections import Counter
all_genres = [genre for genres in df['genres'] for genre in genres]
genre_counts = Counter(all_genres)

df_genres = pd.DataFrame(genre_counts.most_common(15), columns = ['genre', 'count'])
print(df_genres)

              genre  count
0             Drama   1441
1            Comedy   1110
2          Thriller    935
3            Action    918
4         Adventure    661
5           Romance    574
6             Crime    521
7   Science Fiction    431
8            Family    365
9           Fantasy    342
10           Horror    332
11          Mystery    265
12        Animation    188
13          History    145
14              War    120


In [9]:
# ROI promedio por cada genero
df_genre_roi = (
    df.explode('genres').groupby('genres', as_index = False)['roi'].mean().round(2).sort_values('roi', ascending=False).reset_index(drop = True)
)
df_genre_roi.columns = ['genre', 'avg_roi']
print(df_genre_roi.head())

         genre    avg_roi
0       Comedy  766206.62
1        Drama  590702.42
2       Horror  307642.27
3     Thriller  107389.72
4  Documentary   15910.67


In [10]:
# Peliculas por decada
df['decade'] = (df['release_year'] // 10) * 10 # Obtener decadas
movies_per_decade = df.groupby('decade').size().reset_index(name= 'movie_count')
print(movies_per_decade)

    decade  movie_count
0     1910            1
1     1920            3
2     1930           14
3     1940           20
4     1950           25
5     1960           59
6     1970           81
7     1980          204
8     1990          540
9     2000         1335
10    2010          947


In [11]:
# Duracion promedio por decada
avg_runtime = (
    df.groupby('decade')['runtime'].mean().round(1).reset_index().rename(columns={'runtime': 'avg_runtime'})
)
print(avg_runtime)

    decade  avg_runtime
0     1910        197.0
1     1920        134.7
2     1930        111.6
3     1940        115.2
4     1950        121.6
5     1960        137.4
6     1970        119.1
7     1980        111.5
8     1990        112.7
9     2000        108.6
10    2010        109.5


In [13]:
#Top de peliculas mas rentables
top_movies = (df[['title', 'budget', 'revenue', 'net_profit', 'roi', 'release_year']].sort_values('net_profit', ascending = False).head(10).reset_index(drop=True))
print(top_movies)

                                           title     budget     revenue  \
0                                         Avatar  237000000  2787965087   
1                                        Titanic  200000000  1845034188   
2                                 Jurassic World  150000000  1513528810   
3                                      Furious 7  190000000  1506249360   
4                                   The Avengers  220000000  1519557910   
5                        Avengers: Age of Ultron  280000000  1405403694   
6                                         Frozen  150000000  1274219009   
7                                        Minions   74000000  1156730962   
8  The Lord of the Rings: The Return of the King   94000000  1118888979   
9                                     Iron Man 3  200000000  1215439994   

   net_profit      roi  release_year  
0  2550965087  1076.36          2009  
1  1645034188   822.52          1997  
2  1363528810   909.02          2015  
3  1316249360   69

In [14]:
# Conocer correlaciones
num_cols = ['budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count', 'net_profit', 'roi']
correlation = df[num_cols].corr().round(2)
print(correlation)

              budget  revenue  runtime  popularity  vote_average  vote_count  \
budget          1.00     0.71     0.23        0.43         -0.03        0.54   
revenue         0.71     1.00     0.23        0.60          0.19        0.76   
runtime         0.23     0.23     1.00        0.18          0.38        0.26   
popularity      0.43     0.60     0.18        1.00          0.29        0.75   
vote_average   -0.03     0.19     0.38        0.29          1.00        0.38   
vote_count      0.54     0.76     0.26        0.75          0.38        1.00   
net_profit      0.55     0.98     0.21        0.59          0.23        0.74   
roi            -0.02    -0.01    -0.02       -0.00          0.03       -0.00   

              net_profit   roi  
budget              0.55 -0.02  
revenue             0.98 -0.01  
runtime             0.21 -0.02  
popularity          0.59 -0.00  
vote_average        0.23  0.03  
vote_count          0.74 -0.00  
net_profit          1.00 -0.01  
roi            

In [15]:
# Se exportan los resultados a data/processed
df_genres.to_csv('../data/processed/genre_counts.csv', index=False)
df_genre_roi.to_csv('../data/processed/genre_roi.csv', index=False)
movies_per_decade.to_csv('../data/processed/movies_per_decade.csv', index=False)
avg_runtime.to_csv('../data/processed/avg_runtime.csv', index=False)
top_movies.to_csv('../data/processed/top_movies.csv', index=False)
correlation.to_csv('../data/processed/correlation_matrix.csv')
print("Resultados del análisis de datos exportados correctamente")

Resultados del análisis de datos exportados correctamente
